In [15]:
from dotenv import load_dotenv
import os
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from langchain.agents import create_agent

In [8]:
load_dotenv()
api_key = os.getenv("google_api_key")

# **Models**

In [9]:
scene_creator = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=api_key, temperature=0.7)
evalvator = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=api_key, temperature=0.7)

# **Structured output**

In [60]:
class baseScene(BaseModel):
    scene_number: int = Field(description="A simple numeric label used to keep the sequence organized.")
    location: str = Field(description="A short note describing where the scene takes place — could be indoor, outdoor, real place, fictional place, or a general setting.Example: “Inside a small office,” “Busy marketplace,” “Temple courtyard,” “Rural farmland,” etc.")
    time_of_the_day: str = Field(description="Specifies the lighting and mood. Example: “Early morning,” “Sunset,” “Nighttime,” “Midday,” etc.")
    characters_present: str = Field(description="A list of all people (or entities) appearing in the scene. Example: “Main character only,” “Thiruvalluvar,” “Two farmers,” “Student and teacher,” etc.")
    brief_description: str = Field(description="A 1–2 sentence overview of what’s visually happening. Example: “The character walks into the room, observing the surroundings,” or “A peaceful sunrise washes over the village.”")
    dialogue_summary:str = Field(description="A quick line describing what is spoken or conveyed. Example: “The character explains the core idea,” or “Thiruvalluvar delivers a short line about wisdom.”")

class listScenes(BaseModel):
    scenes:List[baseScene] = Field(description="A collection of scene objects, each containing structured details such as scene number, location, time of day, characters, brief description, and dialogue summary. This list represents the full storyboard or sequence of scenes for the video or narrative.")

class evalReport(BaseModel):
    is_approved: bool = Field(description="Boolean value used to define whether the scenes are approved or not")
    correction: str = Field(description="Contains the feedback to improve scenes.")

# **Agents**

In [61]:
scene_agent = create_agent(
    model=scene_creator,
    response_format=listScenes
)

eval_agent = create_agent(
    model=evalvator,
    response_format=evalReport
)

# **1. State**
- Shared data structure that flows through the workflow

In [4]:
class AgentState(TypedDict):
    script: str
    scenes: List[dict]
    eval_result: str
    is_approved: bool
    revision_count: int

# **Graph**

In [78]:
workflow = StateGraph(AgentState)

# **2. Nodes**
- Functions that perform work and update the state
- Each node get state as input and return updated state as output

In [77]:
def scene_creator_agent(state: AgentState):
    """Agent 1: That converts script to scenes"""

    script = state["script"]

    prompt = f"""You are a script breakdown specialist. Convert the following script into scenes and give json prompt.
    
        For each scene, provide:
        - Scene number
        - Location
        - Time of day
        - Characters present
        - Brief description
        - Dialogue summary

        Script:
        {script}

        Format as a structured list of json prompts and give 30 scenes."""
    
    response = scene_agent.invoke({
        "messages": [{"role": "user", "content": prompt}]
    })

    list_of_scenes = [{
        "scene_number": i.scene_number,
        "location": i.location,
        "time_of_day": i.time_of_the_day,
        "characters_present": i.characters_present,
        "brief_description": i.brief_description,
        "dialogue_summary": i.dialogue_summary
    } for i in response["structured_response"].scenes]

    return {
        "scenes": list_of_scenes,
        "revision_count": state.get("revision_count") + 1
    }

def eval_scene_agent(state: AgentState):
    """Agent 2: That evalvates generated scenes and provide feedback if not okay""" 
    scenes = state["scenes"] 
    rules = """
    Rules to check:
    1. Each scene must have a clear location
    2. Each scene must specify time of day
    3. Characters must be clearly identified
    4. Scene transitions must be logical
    5. Each scene should have a clear purpose
    """
    prompt = f"""You are a script supervisor. Evaluate these scenes against the rules.
    
    {rules}

    Scenes to evaluate:
    {scenes}

    Provide:
    1. PASS or FAIL for each rule
    2. Specific issues found
    3. Overall recommendation: APPROVED or NEEDS_REVISION
    4. Suggestions for improvement if needed"""  

    response = eval_agent.invoke({
    "messages": [{"role": "user", "content": prompt}]
    })

    return {
        "is_approved": response["structured_response"].is_approved,
        "eval_result": response["structured_response"].correction
    }

def regenerate_agent(state: AgentState):
    """Agent 3: Regenerate the scenes based on given feedback"""
    scenes = state["scenes"]
    correction = state["eval_result"]

    prompt = f"""Revise these scenes based on the evaluation feedback:

    Original Scenes:
    {scenes}

    Evaluation Feedback:
    {correction}

    Provide improved scenes that address all issues."""

    response = scene_agent.invoke({
    "messages": [{"role": "user", "content": prompt}]
    })

    list_of_scenes = [{
        "scene_number": i.scene_number,
        "location": i.location,
        "time_of_day": i.time_of_the_day,
        "characters_present": i.characters_present,
        "brief_description": i.brief_description,
        "dialogue_summary": i.dialogue_summary
    } for i in response["structured_response"].scenes]

    return {
        "scenes": list_of_scenes,
        "revision_count": state.get("revision_count") + 1
    }

def should_loop(state: AgentState):
    """Decides if revisions need or not"""
    if state["is_approved"]:
        return "end"
    elif state["revision_count"] > 5:
        return "end"
    else:
        return "revise"

In [79]:
workflow.add_node("create_scene", scene_creator_agent)
workflow.add_node("eval_scene", eval_scene_agent)
workflow.add_node("regenerate_scene", regenerate_agent)

# **Edges**
- Connection between nodes
- -> Normal edges: direct connection
- -> Conditional edges: dynamic routing based on state

In [80]:
workflow.add_edge(START, "create_scene")
workflow.add_edge("create_scene", "eval_scene")
workflow.add_conditional_edges("eval_scene", should_loop, {"end": END, "revise": "regenerate_scene"})
workflow.add_edge("regenerate_scene", "eval_scene")

In [81]:
app = workflow.compile()

# **Check**

In [82]:
result = app.invoke(
    {
        "script": "Ever spent hours studying and still remembered nothing the next day? So today, I’m giving you five study tips that actually work, backed by science. First, use the 25–5 rule: study for 25 minutes with zero distractions, then rest for 5 minutes — it keeps your brain fresh. Next, teach what you learned to someone else; if you can explain it simply, you truly understand it, and even talking to a wall works. Use active recall by closing your book and trying to remember the key ideas instead of rereading everything. Keep your study sessions short and consistent because short bursts help you absorb more than long, exhausting sessions. And finally, keep your notes simple with keywords, bullet points, and quick diagrams — simple notes make revision faster. In the end, studying smart always beats studying hard, so keep learning and stay consistent.",
        "scenes": [],
        "eval_result": None,
        "is_approved": None,
        "revision_count": 0
    }
)

In [83]:
result

{'script': 'Ever spent hours studying and still remembered nothing the next day? So today, I’m giving you five study tips that actually work, backed by science. First, use the 25–5 rule: study for 25 minutes with zero distractions, then rest for 5 minutes — it keeps your brain fresh. Next, teach what you learned to someone else; if you can explain it simply, you truly understand it, and even talking to a wall works. Use active recall by closing your book and trying to remember the key ideas instead of rereading everything. Keep your study sessions short and consistent because short bursts help you absorb more than long, exhausting sessions. And finally, keep your notes simple with keywords, bullet points, and quick diagrams — simple notes make revision faster. In the end, studying smart always beats studying hard, so keep learning and stay consistent.',
 'scenes': [{'scene_number': 1,
   'location': 'Modern study room',
   'time_of_day': 'Morning',
   'characters_present': 'Main charac